## Extract localized peak locations for `peaks2`

**Goal**: `localized_peaks_mt_raw.npy` contains one location per entry of `peaks_combined2_sorted.npy`.
We want to extract only the rows that correspond to `peaks2.npy`.

**Pipeline recap**:
- `peaks2` = thresholding → local max → spatial-temporal max
- `peaks_combined2_sorted` = (peaks2 ∪ peaks_matched) → dedup → sort by sample_index
- `localized_peaks_mt_raw` = localized positions of `peaks_combined2_sorted` (1-to-1)

**Strategy**: match each `peaks2` entry to its row in `peaks_combined2_sorted` via the
composite key `(sample_index, channel_index)`, then index into `localized_peaks_mt_raw`.
Peaks2 entries that were removed by the deduplication step are dropped (should be rare).

In [1]:
import numpy as np

BASE = '/ptmp/gongt/GLC7189/GLC7189_g_g1/preprocessed/thresholding_peaks_3.3'

print('Loading arrays...')
p2   = np.load(f'{BASE}/peaks2.npy')
comb = np.load(f'{BASE}/peaks_combined2_sorted.npy')
loc  = np.load(f'{BASE}/localized_peaks_mt_raw.npy')

print(f'peaks2                 : {len(p2):>12,}  fields: {p2.dtype.names}')
print(f'peaks_combined2_sorted : {len(comb):>12,}  fields: {comb.dtype.names}')
print(f'localized_peaks_mt_raw : {len(loc):>12,}  fields: {loc.dtype.names}')
assert len(comb) == len(loc), 'comb and loc must be 1-to-1'

max_ch = int(max(p2['channel_index'].max(), comb['channel_index'].max()))
print(f'max channel_index: {max_ch}')

Loading arrays...
peaks2                 :  102,606,012  fields: ('sample_index', 'channel_index', 'amplitude', 'segment_index', 'z')
peaks_combined2_sorted :  126,258,007  fields: ('sample_index', 'channel_index', 'amplitude', 'segment_index', 'z')
localized_peaks_mt_raw :  126,258,007  fields: ('x', 'y', 'z', 'alpha')
max channel_index: 367


In [2]:
# Build composite keys: (sample_index * multiplier + channel_index)
# multiplier must be > max_ch so the encoding is injective
multiplier = int(max_ch) + 1

print('Building composite keys...')
comb_key = comb['sample_index'].astype(np.int64) * multiplier + comb['channel_index'].astype(np.int64)
p2_key   = p2['sample_index'].astype(np.int64)   * multiplier + p2['channel_index'].astype(np.int64)

# comb is sorted by sample_index, but ties in sample_index may not be sorted by channel_index.
# Sort comb_key and keep track of original positions so we can map back to loc.
print('Sorting comb keys...')
comb_order      = np.argsort(comb_key, kind='stable')   # indices into comb / loc
comb_key_sorted = comb_key[comb_order]

print('Searching...')
# For each peaks2 key find the position it would land in sorted comb_key
ins = np.searchsorted(comb_key_sorted, p2_key)
ins = np.clip(ins, 0, len(comb_key_sorted) - 1)

# Check whether the found position actually matches
matched = comb_key_sorted[ins] == p2_key

n_matched  = matched.sum()
n_dropped  = (~matched).sum()
print(f'peaks2 matched in peaks_combined2_sorted : {n_matched:,} / {len(p2):,}  ({100*n_matched/len(p2):.3f}%)')
print(f'Dropped (removed by dedup step)          : {n_dropped:,}  ({100*n_dropped/len(p2):.4f}%)')

Building composite keys...
Sorting comb keys...
Searching...
peaks2 matched in peaks_combined2_sorted : 97,170,205 / 102,606,012  (94.702%)
Dropped (removed by dedup step)          : 5,435,807  (5.2977%)


In [3]:
# Sanity check: verify a few matched entries look correct
# Pick 5 random matched indices and confirm sample_index + channel_index agree
rng = np.random.default_rng(42)
check_idx = rng.choice(np.where(matched)[0], size=min(5, n_matched), replace=False)
print('Spot-check (peaks2 vs comb entry):')
print(f'{"p2_idx":>8}  {"si_p2":>12}  {"ch_p2":>6}  {"si_comb":>12}  {"ch_comb":>7}  match?')
for i in check_idx:
    comb_i = comb_order[ins[i]]
    ok = (p2['sample_index'][i] == comb['sample_index'][comb_i] and
          p2['channel_index'][i] == comb['channel_index'][comb_i])
    print(f'{i:>8}  {p2["sample_index"][i]:>12}  {p2["channel_index"][i]:>6}  '
          f'{comb["sample_index"][comb_i]:>12}  {comb["channel_index"][comb_i]:>7}  {ok}')

Spot-check (peaks2 vs comb entry):
  p2_idx         si_p2   ch_p2       si_comb  ch_comb  match?
79440983     815125906     214     815125906      214  True
45086155     437340599     255     437340599      255  True
67062251     689009210      53     689009210       53  True
 9177058      83302192     124      83302192      124  True
44476281     431705747     173     431705747      173  True


In [4]:
# Extract locations for matched peaks2 entries
# comb_order[ins[matched]] gives the original indices into comb (and loc)
loc_indices = comb_order[ins[matched]]
loc_peaks2  = loc[loc_indices]

print(f'loc_peaks2 shape : {loc_peaks2.shape}')
print(f'loc_peaks2 dtype : {loc_peaks2.dtype}')
print('First 3 rows:', loc_peaks2[:3])

# Also save the matched peaks2 entries themselves (aligned with loc_peaks2)
# so downstream code has a consistent (peaks, locations) pair
p2_matched = p2[matched]
print(f'p2_matched shape : {p2_matched.shape}')

# Save
out_loc  = f'{BASE}/localized_peaks_mt_raw_peaks2.npy'
out_p2   = f'{BASE}/peaks2_for_motion.npy'   # peaks2 entries that have a location

np.save(out_loc, loc_peaks2)
np.save(out_p2,  p2_matched)

print(f'\nSaved:')
print(f'  {out_loc}')
print(f'  {out_p2}')
print(f'len(loc_peaks2) == len(p2_matched): {len(loc_peaks2) == len(p2_matched)}')

loc_peaks2 shape : (97170205,)
loc_peaks2 dtype : [('x', '<f8'), ('y', '<f8'), ('z', '<f8'), ('alpha', '<f8')]
First 3 rows: [( 82.397323  , 3629.80849248, 108.88416466, 392.96418469)
 ( 26.03942185,  387.32036131,  60.16925958, 241.44206441)
 (-30.95659868, 2539.36257099,  65.24557279, 304.40858793)]
p2_matched shape : (97170205,)

Saved:
  /ptmp/gongt/GLC7189/GLC7189_g_g1/preprocessed/thresholding_peaks_3.3/localized_peaks_mt_raw_peaks2.npy
  /ptmp/gongt/GLC7189/GLC7189_g_g1/preprocessed/thresholding_peaks_3.3/peaks2_for_motion.npy
len(loc_peaks2) == len(p2_matched): True
